### Importar librerías

In [ ]:
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt

from tensorflow.keras.utils import plot_model

### Cargar el dataset MNIST (dígitos escritos a mano)

In [ ]:
# Importar el dataset
mnist_data = keras.datasets.mnist

# Cargar datos de entrenamiento y prueba
(train_images, train_labels), (test_images, test_labels) = mnist_data.load_data()

### Crear nombres de las clases

Aquí las clases son simplemente los dígitos del 0 al 9.

In [ ]:
class_names = [str(i) for i in range(10)]

### Mostrar una imagen de ejemplo (tamaño original 28x28)

In [ ]:
plt.imshow(train_images[7], cmap="gray")
plt.grid(False)
plt.axis("off")
plt.show()

### Verificar el tamaño del dataset

In [ ]:
train_images.shape, test_images.shape

### Distribución de clases

In [ ]:
df = pd.DataFrame(np.unique(train_labels, return_counts=True)).T
df.columns = ["Label", "Count"]
df

### Redimensionar las imágenes a 16x16

El dataset original viene en 28x28, pero para esta entrada trabajamos con imágenes de **16x16 en escala de grises**.
Usamos `tf.image.resize`, que necesita un canal extra (28,28,1) y regresa (16,16,1); al final quitamos ese canal con `squeeze`.

In [ ]:
IMG_SIZE = 16  # tamaño de entrada que vamos a usar

train_images = tf.image.resize(train_images[..., np.newaxis], [IMG_SIZE, IMG_SIZE]).numpy()
test_images = tf.image.resize(test_images[..., np.newaxis], [IMG_SIZE, IMG_SIZE]).numpy()

train_images = train_images.squeeze(-1)
test_images = test_images.squeeze(-1)

train_images.shape, test_images.shape

### Mostrar las primeras 16 imágenes (ya en 16x16)

In [ ]:
plt.figure(figsize=(9,9))

for i in range(16):
    plt.subplot(4,4,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(train_images[i], cmap="gray")
    plt.title(class_names[train_labels[i]])

plt.show()

### Normalizar las imágenes

Los pixeles vienen en escala de grises de 0 a 255. Los normalizamos a un rango de 0 a 1 dividiendo entre 255.

In [ ]:
train_images = train_images / 255.0
test_images = test_images / 255.0

train_images.min(), train_images.max()

### Crear la red neuronal

- **Entrada:** `Flatten`, con la forma dependiendo del número de pixeles (16x16 = 256).
- **Capas ocultas:** `Dense` con activación `relu` (cantidad y neuronas libres).
- **Salida:** `Dense` con 10 neuronas (una por dígito) y activación `softmax`.

In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE)),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(10, activation="softmax")
])

In [ ]:
model.summary()

### Compilar el modelo

- **Loss:** `sparse_categorical_crossentropy`, porque las etiquetas son enteros (0-9) y es un problema multiclase.
- **Optimizer:** `Adam`, con un learning rate ajustable manualmente.

In [ ]:
LEARNING_RATE = 0.001  # libre, se puede ajustar

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

### Ver resumen del modelo

In [ ]:
model.summary()

### Dibujar arquitectura

In [ ]:
plot_model(
    model,
    to_file="model_plot.png",
    show_shapes=True,
    show_layer_names=True
)

### Entrenar el modelo

Usamos `EarlyStopping` monitoreando `val_loss`, para detener el entrenamiento si el modelo deja de mejorar y recuperar los mejores pesos.

In [ ]:
EPOCHS = 30  # libre, EarlyStopping se encarga de frenar antes si ya no mejora

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    train_images,
    train_labels,
    validation_split=0.2,
    epochs=EPOCHS,
    callbacks=[early_stopping]
)

### Graficar la precisión

In [ ]:
plt.plot(history.history["accuracy"])
plt.plot(history.history["val_accuracy"])
plt.title("Model Accuracy")
plt.ylabel("Accuracy")
plt.xlabel("Epoch")
plt.legend(["Train", "Validation"], loc="lower right")
plt.show()

### Graficar la pérdida

In [ ]:
plt.plot(history.history["loss"])
plt.plot(history.history["val_loss"])
plt.title("Model Loss")
plt.ylabel("Loss")
plt.xlabel("Epoch")
plt.legend(["Train", "Validation"], loc="upper right")
plt.show()

### Evaluar el modelo

In [ ]:
test_loss, test_acc = model.evaluate(test_images, test_labels)

print("Test Accuracy:", test_acc)

### Realizar predicciones

In [ ]:
predictions = model.predict(test_images)

### Ver las posibilidades de la primera imagen

In [ ]:
predictions[0].round(2)

### Comparar predicción con etiqueta real

In [ ]:
print("Predicción:", np.argmax(predictions[0]))
print("Etiqueta real:", test_labels[0])

### Mostrar 16 predicciones aleatorias

In [ ]:
figure = plt.figure(figsize=(9,9))

indices = np.random.choice(
    test_images.shape[0],
    size=16,
    replace=False
)

for i, index in enumerate(indices):

    ax = figure.add_subplot(4,4,i+1)

    ax.set_xticks([])
    ax.set_yticks([])

    ax.imshow(test_images[index], cmap="gray")

    predict_index = np.argmax(predictions[index])
    true_index = test_labels[index]

    ax.set_title(
        f"{class_names[predict_index]}\n({class_names[true_index]})",
        color="green" if predict_index == true_index else "red"
    )

plt.tight_layout()
plt.show()

### Guardar el modelo (formato Keras)

Lo guardamos para tenerlo disponible y, en la siguiente celda, convertirlo a TensorFlow.js para poder usarlo desde la GUI web.

In [ ]:
model.save("digit_model_16x16.keras")

### Exportar a TensorFlow.js (para la GUI web)

Instalamos `tensorflowjs` y convertimos el modelo entrenado a un formato que el navegador puede cargar directamente con `tf.loadLayersModel(...)`.
Esto genera una carpeta `tfjs_model/` con un `model.json` y uno o varios archivos `.bin` con los pesos.

**Importante:** copia el contenido completo de `tfjs_model/` dentro de la carpeta `model/` del proyecto web (GUI) antes de subirlo a GitHub/Render.

In [ ]:
!pip install tensorflowjs -q

import tensorflowjs as tfjs

tfjs.converters.save_keras_model(model, "tfjs_model")

import os
os.listdir("tfjs_model")